# Geocoding Historical Place Names (Estonian Columns)

This notebook estimates latitude and longitude for rows that have `country`, `region`, and `district` values in Estonian.

## Important notes

- Yes, it is possible, but historical names can be ambiguous or changed over time.
- We use multiple query variants and keep a confidence flag (`query_used`).
- We do not overwrite existing `lat`/`lon`; we only fill missing values.
- We use OpenStreetMap Nominatim via `geopy` with polite rate limiting.
- A cache file is used so repeated runs are faster and avoid repeated API calls.

In [12]:
from pathlib import Path
import re
import unicodedata

try:
    import pandas as pd
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas'])
    import pandas as pd

try:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'geopy'])
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter

try:
    from tqdm import tqdm
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm'])
    from tqdm import tqdm

try:
    from rapidfuzz import fuzz, process
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rapidfuzz'])
    from rapidfuzz import fuzz, process

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

In [13]:
data_dir = Path('.')
input_file = data_dir / 'finno-ugric-object-places.csv'
output_file = data_dir / 'finno-ugric-object-places-geocoded.csv'
cache_file = data_dir / 'geocode_cache_places.csv'

places = pd.read_csv(input_file)

# Normalize empty strings to missing values for location columns.
for col in ['country', 'region', 'district', 'settlement', 'place_norm', 'lat', 'lon']:
    if col in places.columns:
        places[col] = places[col].replace('', pd.NA)

print(f'Rows: {len(places):,}')
print('Columns:', ', '.join(places.columns))
places.head(3)

Rows: 14,621
Columns: object_id, place_norm, country, region, district, District_controlled, local_admin, settlement, Settlement_controlled, lat, lon, coords_source, place_raw


,object_id,place_norm,country,region,district,District_controlled,local_admin,settlement,Settlement_controlled,lat,lon,coords_source,place_raw
0,518063,Knjazpogostski rajoon; Komi Vabariik; Venemaa,Venemaa,Komi Vabariik,Knjazpogostski rajoon,Knjazpogostski rajoon,NaN,NaN,NaN,NaN,NaN,NaN,"Komi Vabariik, Knjazpogostski rajoon"
1,518064,Knjazpogostski rajoon; Komi Vabariik; Venemaa,Venemaa,Komi Vabariik,Knjazpogostski rajoon,Knjazpogostski rajoon,NaN,NaN,NaN,NaN,NaN,NaN,"Komi Vabariik, Knjazpogostski rajoon"
2,518065,Knjazpogostski rajoon; Komi Vabariik; Venemaa,Venemaa,Komi Vabariik,Knjazpogostski rajoon,Knjazpogostski rajoon,NaN,NaN,NaN,NaN,NaN,NaN,"Komi Vabariik, Knjazpogostski rajoon"


In [17]:
location_columns = ['country', 'region', 'district', 'settlement']

# Add explicit corrections here when you know exact replacements.
manual_overrides = {
    'country': {},
    'region': {},
    'district': {},
    'settlement': {}
}

thresholds = {
    'country': 90,
    'region': 94,
    'district': 96,
    'settlement': 97
}

def normalize_for_match(text):
    if pd.isna(text):
        return None
    text = str(text).strip().lower()
    if not text:
        return None

    # Normalize accents/diacritics for robust fuzzy comparison.
    text = ''.join(
        ch for ch in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(ch)
    )
    text = re.sub(r'[;,/]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def has_same_prefix(a, b, n=3):
    if not a or not b:
        return False
    return a[:n] == b[:n]

def build_spelling_map(series, threshold=95, overrides=None):
    values = (
        series.astype('string')
        .str.strip()
        .replace({'': pd.NA})
        .dropna()
        .value_counts()
        .index
        .tolist()
    )

    overrides = overrides or {}
    mapping = {}
    canonicals = []
    canonical_norms = []

    for value in values:
        if value in overrides:
            mapping[value] = overrides[value]
            continue

        value_norm = normalize_for_match(value)
        if not canonicals:
            canonicals.append(value)
            canonical_norms.append(value_norm)
            mapping[value] = value
            continue

        best = process.extractOne(value_norm, canonical_norms, scorer=fuzz.ratio)
        if best:
            target_norm = canonical_norms[best[2]]
            prefix_ok = has_same_prefix(value_norm, target_norm, n=3)
            if best[1] >= threshold and prefix_ok:
                target = canonicals[best[2]]
                mapping[value] = target
                continue

        canonicals.append(value)
        canonical_norms.append(value_norm)
        mapping[value] = value

    return mapping

change_report = []
mapping_tables = {}

for col in location_columns:
    if col not in places.columns:
        continue

    cleaned_col = f'{col}_clean'
    mapping = build_spelling_map(
        places[col],
        threshold=thresholds[col],
        overrides=manual_overrides.get(col, {})
    )
    places[cleaned_col] = places[col].map(mapping).fillna(places[col])

    changed_mask = places[col].fillna('') != places[cleaned_col].fillna('')
    changed_count = int(changed_mask.sum())
    unique_before = int(places[col].nunique(dropna=True))
    unique_after = int(places[cleaned_col].nunique(dropna=True))

    change_report.append({
        'column': col,
        'unique_before': unique_before,
        'unique_after': unique_after,
        'rows_changed': changed_count
    })

    diff_map = [(k, v) for k, v in mapping.items() if k != v]
    mapping_tables[col] = pd.DataFrame(diff_map, columns=['original', 'cleaned'])

pd.DataFrame(change_report)

,column,unique_before,unique_after,rows_changed
0,country,10,10,0
1,region,91,74,478
2,district,195,195,0
3,settlement,911,891,126


In [19]:
# Preview the first spelling fixes found in each column.
for col in location_columns:
    table = mapping_tables.get(col)
    if table is not None and not table.empty:
        print(f"\nPotential spelling merges in {col} (top 20):")
        display(table.head(20))


Potential spelling merges in region (top 20):


,original,cleaned
0,Bashkiri ANSV,Bashkiiri ANSV
1,Leningrad oblast,Leningradi oblast
2,Peterburgi kubermang,Peterburi kubermang
3,Murmansi oblast,Murmanski oblast
4,Jamali Neenetsi RR,Jamali Neenetsi AR
5,Handi- Mansi AR,Handi-Mansi AR
6,Jamali-Neenetsi AR,Jamali Neenetsi AR
7,Handi- Mansi RR,Handi-Mansi RR
8,Leningardi oblast,Leningradi oblast
9,Kirov oblast,Kirovi oblast



Potential spelling merges in settlement (top 20):


,original,cleaned
0,Koštrõg küla,Kostrõg küla
1,Mordvski Pimburi küla,Mordovski Pimburi küla
2,Kostrõgi küla,Koštrõgi küla
3,Jamali-Neenetsi autonoomne ringkond,Jamali Neenetsi autonoomne ringkond
4,Handi-Mansi Rahvusringkond,Handi-Mansi rahvusringkond
5,Sǟnag küla,Sanag küla
6,Tugijonõ küla,Tugijõno küla
7,Jõgõpera küla,Jõgõperä küla
8,Bolshoje Maresjevo küla,Bolšoje Maresjevo küla
9,Piža küla,Piza küla


## Spelling Normalization Before Geocoding

This step creates cleaned columns for location fields: country, region, district, settlement.

- It uses fuzzy matching to merge likely spelling variants into the most frequent form.
- It keeps original columns unchanged.
- You can add manual corrections in `manual_overrides` for known historical or spelling variants.

In [16]:
def est_clean(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None

def build_query_candidates(row):
    country = est_clean(row.get('country_clean', row.get('country')))
    region = est_clean(row.get('region_clean', row.get('region')))
    district = est_clean(row.get('district_clean', row.get('district')))
    settlement = est_clean(row.get('settlement_clean', row.get('settlement')))
    place_norm = est_clean(row.get('place_norm'))

    candidates = []

    # Most specific to least specific.
    if settlement and district and region and country:
        candidates.append(f"{settlement}, {district}, {region}, {country}")
    if district and region and country:
        candidates.append(f"{district}, {region}, {country}")
    if district and country:
        candidates.append(f"{district}, {country}")
    if region and country:
        candidates.append(f"{region}, {country}")
    if place_norm:
        # place_norm often has semicolon-separated historical variants.
        candidates.append(place_norm.replace(';', ','))
    if country:
        candidates.append(country)

    # Keep order, remove duplicates.
    seen = set()
    ordered = []
    for q in candidates:
        if q not in seen:
            ordered.append(q)
            seen.add(q)
    return ordered

In [7]:
# Load geocode cache if available.
if cache_file.exists():
    cache = pd.read_csv(cache_file)
    cache = cache.dropna(subset=['query']).drop_duplicates(subset=['query'], keep='last')
else:
    cache = pd.DataFrame(columns=['query', 'lat', 'lon', 'address'])

cache_map = {
    row['query']: (row.get('lat'), row.get('lon'), row.get('address'))
    for _, row in cache.iterrows()
}

geolocator = Nominatim(user_agent='finno_ugric_geocoder')
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1, swallow_exceptions=True)

def geocode_with_cache(query):
    if query in cache_map:
        lat, lon, address = cache_map[query]
        if pd.notna(lat) and pd.notna(lon):
            return float(lat), float(lon), address

    location = geocode(query, language='et', addressdetails=True)
    if location is None:
        cache_map[query] = (pd.NA, pd.NA, pd.NA)
        return None

    result = (float(location.latitude), float(location.longitude), location.address)
    cache_map[query] = result
    return result

In [11]:
# Preserve original lat/lon and fill missing values only.
places['lat_filled'] = places['lat']
places['lon_filled'] = places['lon']
places['query_used'] = pd.NA
places['geocoded_address'] = pd.NA

missing_mask = places['lat_filled'].isna() | places['lon_filled'].isna()
to_geocode = places[missing_mask].copy()

print(f"Rows needing geocoding: {len(to_geocode):,}")

for idx, row in tqdm(
    to_geocode.iterrows(),
    total=len(to_geocode),
    desc='Geocoding rows',
    unit='row'
 ):
    queries = build_query_candidates(row)
    found = None
    used_query = None

    for q in queries:
        out = geocode_with_cache(q)
        if out is not None:
            found = out
            used_query = q
            break

    if found is not None:
        lat, lon, address = found
        places.at[idx, 'lat_filled'] = lat
        places.at[idx, 'lon_filled'] = lon
        places.at[idx, 'query_used'] = used_query
        places.at[idx, 'geocoded_address'] = address

print('Geocoding pass complete.')

Rows needing geocoding: 13,812


Geocoding rows:   0%|          | 0/13812 [00:00<?, ?row/s]

Geocoding rows:   6%|▋         | 897/13812 [48:36<11:39:44,  3.25s/row]


KeyboardInterrupt: 

In [ ]:
# Persist cache and enriched output.
cache_out = pd.DataFrame(
    [
        {'query': q, 'lat': vals[0], 'lon': vals[1], 'address': vals[2]}
        for q, vals in cache_map.items()
    ]
)
cache_out.to_csv(cache_file, index=False, encoding='utf-8')

places.to_csv(output_file, index=False, encoding='utf-8')

print(f'Cache saved: {cache_file.name} ({len(cache_out):,} queries)')
print(f'Enriched file saved: {output_file.name}')

In [ ]:
summary = {
    'total_rows': len(places),
    'original_with_coords': int(places['lat'].notna() & places['lon'].notna()).sum(),
    'filled_with_coords': int((places['lat_filled'].notna() & places['lon_filled'].notna()).sum()),
    'newly_geocoded': int(((places['lat'].isna() | places['lon'].isna()) & (places['lat_filled'].notna() & places['lon_filled'].notna())).sum())
}

pd.DataFrame([summary])

In [ ]:
# Inspect a few newly geocoded rows and the exact query used.
new_rows = places[
    (places['lat'].isna() | places['lon'].isna())
    & places['lat_filled'].notna() & places['lon_filled'].notna()
][['object_id', 'country', 'region', 'district', 'settlement', 'query_used', 'lat_filled', 'lon_filled', 'geocoded_address']]

new_rows.head(20)